In [34]:
import requests
from cryptography.hazmat.primitives.serialization import load_pem_public_key
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric import padding
import base64
import textwrap

In [35]:
# Load private key from local file
with open("shashankp.pem", "rb") as key_file:
    private_key = serialization.load_pem_private_key(
        key_file.read(),
        password=None  # Add password=b"your_password" if it's encrypted
    )

print("✅ Private key loaded.")

✅ Private key loaded.


In [36]:
# Download the public key from GitHub
url = "https://raw.githubusercontent.com/shashankp28/shashankp28/refs/heads/main/public.pem"
response = requests.get(url)
public_key_pem = response.content

# Load the public key
public_key = load_pem_public_key(public_key_pem)

print("✅ Public key loaded from GitHub.")

✅ Public key loaded from GitHub.


In [41]:
def create_signed_message_with_pubkey(message: str, name: str, public_key_pem: bytes, url):
    # Wrap message to 80 characters per line
    wrapped_message = textwrap.fill(message, width=80)

    # Step 1: Create the message + key block BEFORE signing
    message_block = f"""
========================= Message Start =========================
{wrapped_message}
                                              - {name}

{public_key_pem.decode().strip()}
Link: {url}
========================= Message End ===========================
""".strip()

    # Step 2: Sign this exact message block
    signature = private_key.sign(
        message_block.encode(),  # sign the formatted, visible message
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.MAX_LENGTH
        ),
        hashes.SHA256()
    )

    # Step 3: Base64 encode and wrap signature
    b64_signature = base64.b64encode(signature).decode()
    wrapped_signature = textwrap.fill(b64_signature, width=80)

    # Step 4: Add wrapped signature
    final_output = f"{message_block}\n\n🔐 Signature (SHA256 + Private Key):\n{wrapped_signature}"
    return final_output

In [42]:
url = "https://pastebin.com/hNP1LT9A"
msg = "This is clue #3: What hides in light, reveals in shadow."
name = "Shashank P"
output = create_signed_message_with_pubkey(msg, name, public_key_pem, url)
with open("cipher.txt", "w") as f:
    f.write(output)